<a href="https://colab.research.google.com/github/JaviAd/datasharing/blob/master/00_Crusher_Feed_Summary_recon.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [26]:
file_paths = [
#2024    'ClientCrusherFeedSummary (1).csv',
#2024    'ClientCrusherFeedSummary (2).csv'
#    'ClientCrusherFeedSummary (3).csv',
#    'ClientCrusherFeedSummary (4).csv',
    'ClientCrusherFeedSummary (5).csv'
#    'ClientCrusherFeedSummary (6).csv',
#    'ClientCrusherFeedSummary (7).csv'
#    'ClientCrusherFeedSummary (8).csv'
]

dpr_file = 'DPR_2025_full.xlsx'

In [27]:
from datetime import datetime
today_date = datetime.now().strftime('%m%d%Y')

In [28]:
import numpy as np
import pandas as pd

processed_dataframes = []

In [29]:
import io

processed_dataframes = []

for file_path in file_paths:
    with open(file_path, 'r', encoding='utf-8') as f:
        lines = f.readlines()

    empty_line_indices = []
    for i, line in enumerate(lines):
        stripped_line = line.strip()

        if not stripped_line or all(char == ',' for char in stripped_line):
            empty_line_indices.append(i)

    if len(empty_line_indices) < 2:
        print(f"Warning: Could not find at least two empty lines to delimit tables in '{file_path}'. Skipping file.")
        continue

    first_empty_row_index = empty_line_indices[0]
    second_empty_row_index = empty_line_indices[1]

    header_row_index = first_empty_row_index + 1

    table_lines = [lines[header_row_index]] + lines[header_row_index + 1:second_empty_row_index]

    table_data = io.StringIO(''.join(table_lines))

    df_second_table = pd.read_csv(table_data, header=0, engine='python', sep=',')

    df_second_table = df_second_table.reset_index(drop=True)

    processed_dataframes.append(df_second_table)

In [30]:
filtered_dataframes = []

for df in processed_dataframes:
    df_filtered = df.iloc[:, 18:41].copy()

    df_filtered.columns = df_filtered.columns.astype(str)

    column_to_drop = 'tftonnesvalue_group2'

    if column_to_drop in df_filtered.columns:
        df_filtered = df_filtered.drop(columns=[column_to_drop])

    filtered_dataframes.append(df_filtered)

In [31]:
final_combined_df = pd.concat(filtered_dataframes, ignore_index=True)

column_rename_mapping = {
    "MaterialName_group2": "Material",
    "payloadvalue_group2": "payload",
    "auvalue_group2": "Au",
    "agvalue_group2": "Ag",
    "CuValue_group2": "Cu",
    "S_value_group2": "S_pct",
    "hgvalue_group2": "Hg",
    "ZnValue_group2": "Zn",
    "kValue_group2": "k",
    "CValue_group2": "C",
    "S2Value_group2": "S2",
    "HardnesValue_group2": "Hardness",
    "lithoValue_group2": "litho",
    "aurValue_group2": "AuR",
    "agrValue_group2": "AgR",
    "cur_value_group2": "CuR",
    "mettypeValue_group2": "Mettype",
    "oc_value_group2": "OC"
}

final_combined_df = final_combined_df.rename(columns=column_rename_mapping)

# Remove rows where 'Grade' starts with 'SJ' as requested
final_combined_df = final_combined_df[~final_combined_df['Grade'].astype(str).str.startswith('SJ')]

# Replace all values in 'payload' column with 182
final_combined_df['payload'] = 182

In [32]:
final_combined_df['TimeEmpty'] = pd.to_datetime(final_combined_df['TimeEmpty'], errors='coerce')

# Shift cutoff time (6:30 AM)
cutoff_time = pd.to_datetime('06:30:00').time()

def get_custom_date(timestamp):
    if pd.isna(timestamp):
        return pd.NaT
    if timestamp.time() < cutoff_time:
        return (timestamp - pd.Timedelta(days=1)).normalize()
    else:
        return timestamp.normalize()

final_combined_df['CustomDate'] = final_combined_df['TimeEmpty'].apply(get_custom_date)

# Format the 'CustomDate' column to mm/dd/yyyy string
#final_combined_df['CustomDate_Formatted'] = final_combined_df['CustomDate'].dt.strftime('%m/%d/%Y')

final_combined_df.to_csv(f'00. Crusher_feed_{today_date}.csv', index=False)

In [33]:
aggregation_dict = {
    'payload': 'sum',
    'Au': 'mean',
    'Ag': 'mean',
    'Cu': 'mean',
    'S_pct': 'mean',
    'Hg': 'mean',
    'Zn': 'mean',
    'k': 'mean',
    'C': 'mean',
    'S2': 'mean',
    'Hardness': 'mean',
    'litho': 'mean',
    'AuR': 'mean',
    'AgR': 'mean',
    'CuR': 'mean',
    'OC': 'mean'
}

pivot_table_main = final_combined_df.groupby(['CustomDate', 'Grade', 'Material', 'Mettype']).agg(aggregation_dict)
pivot_table_grade_subtotals = final_combined_df.groupby(['CustomDate', 'Grade']).agg(aggregation_dict)
mettype_counts = final_combined_df.groupby(['CustomDate', 'Grade'])['Mettype'].nunique()
grades_with_multiple_mettypes = mettype_counts[mettype_counts > 1].index
pivot_table_grade_subtotals_filtered = pivot_table_grade_subtotals.loc[grades_with_multiple_mettypes]
pivot_table_grade_subtotals_filtered['Material'] = 'TOTAL_MATERIAL'
pivot_table_grade_subtotals_filtered['Mettype'] = 'TOTAL_GRADE'
pivot_table_grade_subtotals_filtered = pivot_table_grade_subtotals_filtered.reset_index().set_index(['CustomDate', 'Grade', 'Material', 'Mettype'])

pivot_table_combined = pd.concat([pivot_table_main, pivot_table_grade_subtotals_filtered])

pivot_table_final = pivot_table_combined.sort_values(by=['CustomDate', 'Grade', 'Material', 'Mettype'])
pivot_table_final = pivot_table_final.reset_index()

daily_total_payload = final_combined_df.groupby('CustomDate')['payload'].sum().rename('Daily_Total_payload')

pivot_table_final = pd.merge(pivot_table_final, daily_total_payload, on='CustomDate', how='left')

pivot_table_final['Pct day'] = (pivot_table_final['payload'] / pivot_table_final['Daily_Total_payload']) * 100

pivot_table_final = pivot_table_final.drop(columns=['Daily_Total_payload'])

In [34]:
finger_stockpile_list = ['F-LG1A/285-A001',
                         'F-LG1L/350-A015',
                         'F-LG1L/360-A002',
                         'F-LG1L/360-D002',
                         'F-LG2B/290-B011',
                         'F-LG2B/290-B012',
                         'F-LG2B/290-B013',
                         'F-LG2B/290-B014',
                         'F-LG2B/290-B015',
                         'F-LG2B/290-B016',
                         'F-LG2B/290-B017',
                         'F-LG2B/290-B018',
                         'F-LG2B/290-B019',
                         'F-LG2B/290-B020',
                         'F-LG2B/290-B021',
                         'F-LG2B/290-B022',
                         'F-LG2B/290-B023',
                         'F-LG2B/290-B024',
                         'F-LG2B/290-B025',
                         'F-LG2B/290-B026',
                         'F-LG2L/350-A012',
                         'F-LG2M/290-D001',
                         'F-LG2R/300-A002',
                         'F-LG2R/300-A003',
                         'F-LG3D/407-A003',
                         'F-LG3D/407-B001',
                         'F-LG3D/407-B002',
                         'F-LG3D/407-B003',
                         'F-LG3D/407-B004',
                         'F-LG3F/420-A001',
                         'F-LG3L/350-A011',
                         'F-LG3L/360-C001',
                         'F-LG3L/360-C002',
                         'F-LG3L/360-G001',
                         'F-LG3L/360-G002',
                         'F-LG3L/360-G003',
                         'F-LG3L/360-G004',
                         'F-LG3L/360-G005',
                         'F-MG3D/407-A001',
                         'F-MG3D/407-A002',
                         'F-MG3D/407-A003'
                         #F-HG1L/360-F003 Stockpile or fresh? No data
                         #Updated 2025 full to March 31st, 2026
                         ]

pivot_table_final['Type'] = pivot_table_final['Grade'].apply(lambda x: 'Stockpile' if str(x).startswith('SP') else ('FR Finger' if str(x).startswith('F') else 'Fresh'))
pivot_table_final['Origen'] = pivot_table_final['Grade'].apply(lambda x: str(x).rsplit('/', 1)[0] if '/' in str(x) else x)
pivot_table_final['Origen_merge'] = pivot_table_final.apply(lambda row: row['Origen'] if row['Type'] == 'Stockpile' else 'Fresh Material', axis=1)

pivot_table_final.loc[pivot_table_final['Origen'].isin(finger_stockpile_list), 'Type'] = 'SP Finger'
pivot_table_final.loc[pivot_table_final['Origen'].isin(finger_stockpile_list), 'Origen_merge'] = 'SP Finger'

pivot_table_final['Stock'] = pivot_table_final.apply(lambda row: str(row['Grade'])[:6] if row['Type'] == 'Stockpile' else '', axis=1)
pivot_table_final['Stock_fase'] = pivot_table_final.apply(lambda row: str(row['Grade'])[5] if row['Type'] == 'Stockpile' and len(str(row['Grade'])) > 5 else '', axis=1)

pivot_table_final.to_csv(f'01. Crusher_feed_pivot_{today_date}.csv', index=False)

In [35]:
# Process data from Daily Production Report
df_raw = pd.read_excel(dpr_file, header=None)

date_headers = df_raw.iloc[4, 9:].copy()

cleaned_date_headers = []
name_counts = {}
for header in date_headers:
    original_header = str(header)
    count = name_counts.get(original_header, 0)
    if count > 0:
        cleaned_date_headers.append(f"{original_header}_{count}")
    else:
        cleaned_date_headers.append(original_header)
    name_counts[original_header] = count + 1

row_labels = df_raw.iloc[11:, 1].copy()

cleaned_row_labels = []
name_counts = {}
for label in row_labels:
    original_label = str(label)
    count = name_counts.get(original_label, 0)
    if count > 0:
        cleaned_row_labels.append(f"{original_label}_{count}")
    else:
        cleaned_row_labels.append(original_label)
    name_counts[original_label] = count + 1

df_data = df_raw.iloc[11:, 9:]

df_processed = pd.DataFrame(df_data.values, columns=cleaned_date_headers, index=cleaned_row_labels)

In [36]:
# Convertir los nombres de las columnas a formato de fecha (MM/DD/YYYY)
new_columns_formatted = []
for col in df_processed.columns:
    try:
        dt_obj = pd.to_datetime(col)
        new_columns_formatted.append(dt_obj.strftime('%m/%d/%Y'))
    except ValueError:
        new_columns_formatted.append(col)

df_processed.columns = new_columns_formatted

df_transposed = df_processed.T

display(df_transposed.head())

,Overall Mill Production,Dry Milled Total Tonnes,Au Mill Feed,Ag Mill Feed,S-2 Mill Feed,TC Mill Feed,OC Mill Feed,SAG Mill Production,SAG Mill Total Tonnes,Au SAG Mill Feed,...,Grade Tailings (Rolling 7 days Average),Gold Lost to Tailings,Silver Lost to Tailings,Plant Tails 1,Plant Tails 2,Tonnage to Tailings by CIL2 Bypass,Gold lost from CIL 2 Bypass (oz),Silver lost from CIL Bypass (oz),Total HDS Gold Losses (oz),Total HDS Silver Losses (oz)
01/01/2025,NaN,29211.78,1.984687,14.207809,0.05689,0.000946,0.000535,NaN,20620,1.947306,...,NaN,233.953846,5220.44345,NaN,NaN,0,0,0,24.188153,82.26436
01/02/2025,NaN,30755.26,2.014073,13.784515,0.054999,0.000692,0.000494,NaN,17719.7,1.987681,...,NaN,269.212913,5501.947934,NaN,NaN,883.34,25.778778,121.8369,41.338652,345.473427
01/03/2025,NaN,25725.02,2.092393,14.466811,0.049609,0.000236,0.000921,NaN,775.01,1.919041,...,NaN,138.825145,6071.287638,NaN,NaN,0,0,0,20.179894,40.396693
01/04/2025,NaN,29972.11,1.895069,14.122207,0.055112,0.000905,0.000735,NaN,19020.48,1.848954,...,NaN,205.718269,6973.720631,NaN,NaN,0,0,0,57.876599,376.719871
01/05/2025,NaN,28913.52,1.988861,14.78902,0.051584,0.001746,0.000706,NaN,25061.62,1.981284,...,NaN,161.665156,5880.999423,NaN,NaN,0,0,0,36.554174,27.475703


In [37]:
df_final_processed = df_transposed.dropna(axis=1, how='all')
df_final_processed = df_final_processed.reset_index()
df_final_processed = df_final_processed.rename(columns={'index': 'Date'})
current_columns = df_final_processed.columns.tolist()

new_names_for_14_cols = [
    "Overall Au Recovery pc", "CIL1 Au Recovery pc", "CIL2 Au Recovery pc",
    "Overall Ag Recovery pc", "CIL1 Ag Recovery pc", "CIL2 Ag Recovery pc",
    "Overall Au Recovered oz", "CIL1 Au Recovered oz", "CIL2 Au Recovered oz",
    "Overall Ag Recovered oz", "CIL1 Ag Recovered oz", "CIL2 Ag Recovered oz",
    "Au poured", "Ag Poured"
]

# Encontrar el índice de la columna de inicio para el renombramiento
# La columna de partida es 'Gold'
try:
    start_index = current_columns.index('Gold')
except ValueError:
    print("Error: La columna 'Gold' no se encontró en el DataFrame.")
    # Asignar un valor que indique que no se encontró para evitar errores posteriores
    start_index = -1

if start_index != -1 and (start_index + 14) <= len(current_columns):
    columns_to_rename_actual = current_columns[start_index : start_index + 14]

    rename_mapping = dict(zip(columns_to_rename_actual, new_names_for_14_cols))

    df_final_processed = df_final_processed.rename(columns=rename_mapping)
    print("Columnas renombradas con éxito.")
else:
    print("Advertencia: No se pudieron renombrar las 14 columnas. El índice de inicio no es válido o no hay suficientes columnas después del punto de partida.")

df_final_processed.to_csv(f'02. DPR_processed_{today_date}.csv', index=False) # index=False porque 'Date' ya es una columna

Columnas renombradas con éxito.


In [38]:
# Asegurarse de que las columnas de fecha estén en formato datetime para una unión correcta
final_combined_df['CustomDate'] = pd.to_datetime(final_combined_df['CustomDate'])
pivot_table_final['CustomDate'] = pd.to_datetime(pivot_table_final['CustomDate'])
df_final_processed['Date'] = pd.to_datetime(df_final_processed['Date'])

merged_df = pd.merge(pivot_table_final, df_final_processed, left_on='CustomDate', right_on='Date', how='left')

if 'Date' in merged_df.columns:
    merged_df = merged_df.drop(columns=['Date'])

merged_df.to_csv(f'03. Crusher_feed_pivot_full_{today_date}.csv', index=False)

In [39]:
pivot_table_final_columns = [
    'CustomDate',
    'Grade',
    'Material',
    'Mettype',
    'payload',
    'Au',
    'Ag',
    'Cu',
    'S_pct',
    'Hg',
    'Zn',
    'k',
    'C',
    'S2',
    'Hardness',
    'litho',
    'AuR',
    'AgR',
    'CuR',
    'OC',
    'Pct day',
    'Type',
    'Origen',
    'Origen_merge',
    'Stock',
    'Stock_fase'
]

dpr_specific_columns = [
    "Dry Milled Total Tonnes",
    "Flotation Feed Tonnes",
    "Flotation Tails Tonnes",
    "Autoclave Tonnes",
    "CIL 1 Tonnes",
    "CIL 2 Tonnes",
    "Overall Au Recovery pc",
    "CIL1 Au Recovery pc",
    "CIL2 Au Recovery pc"
]

# Combine the lists of columns
columns_for_export = pivot_table_final_columns + dpr_specific_columns

# Create the subset DataFrame
export_df = merged_df[columns_for_export].copy()

# Export the subset DataFrame to a CSV file
export_df.to_csv(f'04. Crusher_feed_subset_{today_date}.csv', index=False)

In [40]:
recon_columns = [
    'CustomDate', 'Grade', 'Material', 'Mettype', 'payload', 'Au', 'Ag', 'Cu',
    'S_pct', 'C', 'S2', 'OC', 'Type', 'Dry Milled Total Tonnes','Flotation Feed Tonnes',
    'Flotation Concentrate Tonnes', 'Sulfide Recovery', 'Gold Recovery',
    'Autoclave Tonnes', 'CIL 2 Tonnes', 'Overall Au Recovery pc', 'CIL1 Au Recovery pc',
    'CIL2 Au Recovery pc'
]

Recon_df = merged_df[recon_columns].copy()

In [41]:
# Convert columns to numeric, coercing errors to NaN
Recon_df['Flotation Feed Tonnes'] = pd.to_numeric(Recon_df['Flotation Feed Tonnes'], errors='coerce')
Recon_df['Autoclave Tonnes'] = pd.to_numeric(Recon_df['Autoclave Tonnes'], errors='coerce')
Recon_df['Flotation Concentrate Tonnes'] = pd.to_numeric(Recon_df['Flotation Concentrate Tonnes'], errors='coerce')

# Fill NaN values with 0 before calculation (no tonnage)
Recon_df['Flotation Feed Tonnes'] = Recon_df['Flotation Feed Tonnes'].fillna(0)
Recon_df['Autoclave Tonnes'] = Recon_df['Autoclave Tonnes'].fillna(0)
Recon_df['Flotation Concentrate Tonnes'] = Recon_df['Flotation Concentrate Tonnes'].fillna(0)

Recon_df['Flot_Split'] = Recon_df['Flotation Feed Tonnes'] / (Recon_df['Flotation Feed Tonnes'] + Recon_df['Autoclave Tonnes'] - Recon_df['Flotation Concentrate Tonnes'])


In [42]:
# Ensure 'S2' and 'Au' columns are numeric, coercing errors to NaN
Recon_df['S2'] = pd.to_numeric(Recon_df['S2'], errors='coerce')
Recon_df['Au'] = pd.to_numeric(Recon_df['Au'], errors='coerce')

# Handle potential NaN values in 'S2' and 'Au' by filling with 0 or a small number
# Filling 'Au' with a small non-zero number to prevent division by zero
Recon_df['S2'] = Recon_df['S2'].fillna(0)
Recon_df['Au'] = Recon_df['Au'].fillna(0.0001) # Avoid division by zero

# Define conditions and choices
conditions = [
    (Recon_df['Type'].isin(['Stockpile', 'SP Finger'])) &
    (Recon_df['Mettype'].isin(['MO-VCL', 'MN-VCL', 'MN-SP'])) &
    (Recon_df['Material'].isin(['L01', 'Min. Waste PAG'])),

    (Recon_df['Type'].isin(['Stockpile', 'SP Finger'])) &
    (Recon_df['Mettype'].isin(['MO-VCL', 'MN-VCL', 'MN-SP'])) &
    (Recon_df['Material'] == 'L02'),

    (Recon_df['Type'].isin(['Stockpile', 'SP Finger'])) &
    (Recon_df['Mettype'].isin(['MO-VCL', 'MN-VCL', 'MN-SP'])) &
    (Recon_df['Material'] == 'L03'),

    (Recon_df['Type'].isin(['Stockpile', 'SP Finger'])) &
    (Recon_df['Mettype'].isin(['MO-BSD', 'MN-BSD', 'UNK'])) &
    (Recon_df['Material'].isin(['L01', 'L02'])),

    (Recon_df['Type'].isin(['Stockpile', 'SP Finger'])) &
    (Recon_df['Mettype'].isin(['MO-BSD', 'MN-BSD', 'UNK'])) &
    (Recon_df['Material'] == 'L03'),

    (Recon_df['Type'].isin(['Fresh', 'FR Finger'])) &
    (Recon_df['Mettype'].isin(['MO-BSD', 'MN-BSD'])),

    (Recon_df['Type'].isin(['Fresh', 'FR Finger'])) &
    (Recon_df['Mettype'].isin(['MO-VCL', 'MN-VCL'])),

    (Recon_df['Type'].isin(['Fresh', 'FR Finger'])) &
    (Recon_df['Mettype'] == 'MN-SP')
]

choices = [
    0.741 - 0.035,
    0.787 - 0.035,
    0.763 - 0.035,
    0.767 - 0.035,
    0.744 - 0.035,
    (Recon_df['S2'] * 0.586 + 0.202 * Recon_df['S2'] / Recon_df['Au'] - 7.5 / Recon_df['Au'] + 78.25) / 100 - 0.035,
    (Recon_df['S2'] * 0.184 + 0.202 * Recon_df['S2'] / Recon_df['Au'] - 7.5 / Recon_df['Au'] + 93.18) / 100 - 0.035,
    (Recon_df['S2'] * 0.152 + 0.139 * Recon_df['S2'] / Recon_df['Au'] - 5.18 / Recon_df['Au'] + 94.38) / 100 - 0.035
]

# Create the new column using np.select
Recon_df['Flot_Rec_calc'] = np.select(conditions, choices, default=np.nan)


In [43]:
# Define conditions for 'POX_Rec_calc'
conditions_pox = [
    (Recon_df['Type'].isin(['Fresh', 'FR Finger'])) & (Recon_df['Au'] > 3.33),
    (Recon_df['Type'].isin(['Fresh', 'FR Finger'])) & (Recon_df['Au'] <= 3.33),
    (Recon_df['Type'].isin(['Stockpile', 'SP Finger'])) & Recon_df['Mettype'].isin(['MO-VCL', 'MN-VCL', 'MN-SP']) & Recon_df['Material'].isin(['L01', 'Min. Waste PAG']),
    (Recon_df['Type'].isin(['Stockpile', 'SP Finger'])) & Recon_df['Mettype'].isin(['MO-VCL', 'MN-VCL', 'MN-SP']) & (Recon_df['Material'] == 'L02'),
    (Recon_df['Type'].isin(['Stockpile', 'SP Finger'])) & Recon_df['Mettype'].isin(['MO-VCL', 'MN-VCL', 'MN-SP']) & (Recon_df['Material'] == 'L03'),
    (Recon_df['Type'].isin(['Stockpile', 'SP Finger'])) & Recon_df['Mettype'].isin(['MO-BSD', 'MN-BSD', 'UNK']) & Recon_df['Material'].isin(['L01', 'L02']),
    (Recon_df['Type'].isin(['Stockpile', 'SP Finger'])) & Recon_df['Mettype'].isin(['MO-BSD', 'MN-BSD', 'UNK']) & (Recon_df['Material'] == 'L03')
]

# Define choices for 'POX_Rec_calc'
choices_pox = [
    0.9,
    (0.9 * Recon_df['Au'] + 87) / 100,
    0.879,
    0.819,
    0.823,
    0.87,
    0.742
]

# Create the new column 'POX_Rec_calc' using np.select
Recon_df['POX_Rec_calc'] = np.select(conditions_pox, choices_pox, default=np.nan)


In [44]:
Recon_df['Overall_Rec_calc'] = np.nan

# Define the condition for when the calculation should be applied
valid_calculation_condition = Recon_df['Flot_Rec_calc'].notna() & Recon_df['POX_Rec_calc'].notna()

# Apply the calculation only where both 'Flot_Rec_calc' and 'POX_Rec_calc' are not NaN
Recon_df.loc[valid_calculation_condition, 'Overall_Rec_calc'] = \
    (1 - Recon_df['Flot_Split']) * Recon_df['POX_Rec_calc'] + \
    Recon_df['Flot_Split'] * Recon_df['Flot_Rec_calc'] * Recon_df['POX_Rec_calc'] + \
    Recon_df['Flot_Split'] * (1 - Recon_df['Flot_Rec_calc']) * 0.34

Recon_df.to_csv(f'05. Recon_df_output_{today_date}.csv', index=False)

In [45]:
Recon_df = Recon_df.rename(columns={
    "Gold Recovery": "Flot_Rec_act",
    "CIL1 Au Recovery pc": "POX_Rec_act",
    "Overall Au Recovery pc": "Overall_Rec_act"
})


In [46]:
# Define a function for weighted average
def weighted_average(df, data_col, weight_col):
    # Ensure both columns are numeric and handle potential NaN values
    df[data_col] = pd.to_numeric(df[data_col], errors='coerce')
    df[weight_col] = pd.to_numeric(df[weight_col], errors='coerce')

    # Drop rows where either data_col or weight_col is NaN, or weight_col is 0
    temp_df = df.dropna(subset=[data_col, weight_col])
    temp_df = temp_df[temp_df[weight_col] != 0]

    if temp_df.empty: # If no valid data for weighted average
        return np.nan
    return (temp_df[data_col] * temp_df[weight_col]).sum() / temp_df[weight_col].sum()

# Define a function to apply to each group
def custom_group_agg(group_df):
    results = {}

    # Weighted averages for relevant columns
    weighted_avg_cols = [
        'Flot_Rec_act', 'POX_Rec_act', 'Overall_Rec_act',
        'Flot_Rec_calc', 'POX_Rec_calc', 'Overall_Rec_calc',
        'Flotation Feed Tonnes', 'Flotation Concentrate Tonnes',
        'Autoclave Tonnes', 'CIL 2 Tonnes'
    ]
    for col in weighted_avg_cols:
        results[col] = weighted_average(group_df, col, 'payload')

    # Sum aggregations for tonnage columns
    sum_cols = [
        'payload'
    ]
    for col in sum_cols:
        results[col] = group_df[col].sum()

    return pd.Series(results)

# Perform the groupby and aggregation using apply
pivot_daily_df = Recon_df.groupby('CustomDate').apply(custom_group_agg, include_groups=False).reset_index()

# Apply the user-requested validations
pivot_daily_df.loc[pivot_daily_df['Flotation Feed Tonnes'] == 0, 'Flot_Rec_act'] = np.nan
pivot_daily_df.loc[pivot_daily_df['Autoclave Tonnes'] == 0, 'POX_Rec_act'] = np.nan
pivot_daily_df.loc[(pivot_daily_df['Flotation Feed Tonnes'] == 0) & (pivot_daily_df['Autoclave Tonnes'] == 0), 'Overall_Rec_act'] = np.nan

pivot_daily_df.to_csv(f'06. Recon_pivot_{today_date}.csv', index=False)